# LogHAR Model on the MHAR Feature Set

**Purpose:** The purpose of this notebook is to compute the out-of-sample forecast performance 
of the Log-HAR (LogHAR) model of Corsi (2009) on the $\mathcal{M}_{\mathrm{HAR}}$ 
feature set for the EURO STOXX 50 index over the period 1999 - 2020, following the 
methodology of Christensen et al. (2023). We evaluate forecast 
accuracy using the Mean Squared Error (MSE) and report the relative MSE with 
respect to the HAR benchmark. The LogHAR model extends the baseline HAR by 
applying a log-transformation to both the dependent variable and all predictors, 
which imposes a nonlinear relationship between present and future volatility and 
attenuates the influence of outliers — a key advantage given the heavily 
right-skewed distribution of realized variance.

The model is defined as:

$$\log(RV_{t+1}) = \beta_0 + \beta_d \log(RV_t) + \beta_w \log\left(\overline{RV}_t^{(w)}\right) + 
\beta_m \log\left(\overline{RV}_t^{(m)}\right) + \varepsilon_{t+1}$$

where the predictors are the log-transformed current day's realized variance 
measures:

- $\log(RV_t)$ (logRVD) — log of daily realized variance
- $\log\left(\overline{RV}_t^{(w)}\right) = \log\left(\frac{1}{5}\sum_{i=0}^{4} 
RV_{t-i}\right)$ (logRVW) — log of 5-day rolling average of realized variance
- $\log\left(\overline{RV}_t^{(m)}\right) = \log\left(\frac{1}{22}\sum_{i=0}^{21} 
RV_{t-i}\right)$ (logRVM) — log of 22-day rolling average of realized variance

Since the LogHAR produces forecasts of $\log(RV_{t+1})$ rather than $RV_{t+1}$ 
directly, the forecasts must be back-transformed to the original scale prior to 
evaluation. Following Christensen et al. (2023), we apply a bias correction to 
account for Jensen's inequality, which arises because the expectation of a 
nonlinear transformation is not equal to the nonlinear transformation of the 
expectation:

$$\widehat{RV}_{t+1} = \exp\left(\hat{f}(Z_t) + \frac{1}{2}\hat{\sigma}^2\right)$$

where $\hat{f}(Z_t)$ is the forecast of $\log(RV_{t+1})$ and $\hat{\sigma}^2$ 
is the variance of the residuals from the training and validation set. This 
bias correction is appropriate when the distribution of log-realized variance 
is approximately Gaussian, which is well-documented in the literature 
\citep{andersen2001distribution, christensen2019realized}.

The model is estimated via Ordinary Least Squares (OLS) using a rolling window 
of fixed length equal to the combined training and validation set (4,389 
observations), rolling forward one day at a time across the test period of 
1,099 observations (2016-04-20 to 2020-08-05).

## 1.1 Loading Data

We load the MALL dataset which contains all predictor variables and both target 
variables for the EURO STOXX 50 index over the period 1999–2020. For the LogHAR 
model we use the log-transformed target variable $\log(RV_{t+1})$, represented 
by \texttt{logRV\_target} in our dataset, and the log-transformed predictors 
$\texttt{logRVD}$, $\texttt{logRVW}$, and $\texttt{logRVM}$. The log-transformed 
target variable was constructed as $\log(RV_{t+1})$ and saved alongside the 
original level target variable $RV_{t+1}$ in the MALL dataset.

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings("ignore")

# Load MALL dataset
file_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/MALL.csv"
MALL = pd.read_csv(file_path, index_col='Date', parse_dates=True)

print(f"Shape: {MALL.shape}")
print(f"Date range: {MALL.index[0].date()} to {MALL.index[-1].date()}")
print(f"NaN values: {MALL.isna().sum().sum()}")

# Verify log-transformed variables and target are present
print(f"\nFeature set verification:")
print(f"  logRVD present:      {'logRVD'      in MALL.columns}")
print(f"  logRVW present:      {'logRVW'      in MALL.columns}")
print(f"  logRVM present:      {'logRVM'      in MALL.columns}")
print(f"  logRV_target present: {'logRV_target' in MALL.columns}")

# Preview relevant columns
print(f"\nFirst few rows of LogHAR variables:")
print(MALL[["logRVD", "logRVW", "logRVM", 
            "RV_target", "logRV_target"]].head())

Shape: (5489, 31)
Date range: 1999-01-05 to 2020-08-06
NaN values: 0

Feature set verification:
  logRVD present:      True
  logRVW present:      True
  logRVM present:      True
  logRV_target present: True

First few rows of LogHAR variables:
              logRVD    logRVW    logRVM  RV_target  logRV_target
Date                                                             
1999-01-05 -9.462355 -9.288024 -8.958306   0.000058     -9.760055
1999-01-06 -9.760055 -9.290171 -9.005038   0.000139     -8.877543
1999-01-07 -8.877543 -9.192627 -9.023153   0.000108     -9.130603
1999-01-08 -9.130603 -9.148418 -9.097084   0.000166     -8.702667
1999-01-11 -8.702667 -9.116188 -9.115384   0.000115     -9.067553


#### Creating LogRV_target variable for LogHAR Model